In [2]:
# Cell 1
from pathlib import Path
import json
from copy import deepcopy


DEFAULT_RULES = {
    "deep_depth": 200,
    "shallow_depth": 50,
    "initial_temp": 0.15,
    "noise_frac": 0.1,
    "temp_decay_turn": 10,
    "use_early_resign": False,
    "use_fast_finish": False,
    "use_override_temp": False,
    "fixed_komi": 0.0,
    "use_pcr": False,
}


In [3]:
# Cell 2
def get_predecessors(items, index, history=10):
    return items[max(0, index - history):index]


def make_rules(base_rules=None, simulation_budget=None, initial_temp=None, temp_decay_turn=None):
    rules = deepcopy(base_rules or DEFAULT_RULES)
    if simulation_budget is not None:
        rules["deep_depth"] = simulation_budget
    if initial_temp is not None:
        rules["initial_temp"] = initial_temp
    if temp_decay_turn is not None:
        rules["temp_decay_turn"] = temp_decay_turn
        
    return rules


def make_entry(agent_a, agent_b, rules_a, rules_b, total_games=800):
    return {
        "agent_a": agent_a,
        "agent_b": agent_b,
        "total_games": total_games,
        "rules_a": rules_a,
        "rules_b": rules_b,
    }


def preview_schedule(schedule, limit=20):
    for e in schedule[:limit]:
        print(
            f"{e['agent_a']:20s} vs {e['agent_b']:20s} | "
            f"budget_a={e['rules_a']['deep_depth']:>3} | "
            f"budget_b={e['rules_b']['deep_depth']:>3} | "
            f"games={e['total_games']}"
        )


def save_schedule_jsonl(schedule, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for entry in schedule:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Сохранено: {path}")


def summarize_schedule(schedule):
    print(f"Итого матчей: {len(schedule)}")
    print(f"Итого игр: {sum(e['total_games'] for e in schedule)}")


In [4]:
# Cell 3
def build_agent_schedule(
    agents,
    simulation_budget=200,
    history=10,
    total_games=800,
    base_rules=None,
):
    schedule = []

    common_rules = make_rules(
        base_rules=base_rules,
        simulation_budget=simulation_budget,
    )

    for i, agent_b in enumerate(agents):
        predecessors = get_predecessors(agents, i, history=history)
        for agent_a in predecessors:
            entry = make_entry(
                agent_a=agent_a,
                agent_b=agent_b,
                rules_a=deepcopy(common_rules),
                rules_b=deepcopy(common_rules),
                total_games=total_games,
            )
            schedule.append(entry)

    return schedule

In [5]:
# Cell 4
def build_budget_schedule(
    agent_name,
    budgets,
    history=10,
    total_games=800,
    base_rules=None,
):
    schedule = []

    for i, budget_b in enumerate(budgets):
        predecessors = get_predecessors(budgets, i, history=history)

        for budget_a in predecessors:
            entry = make_entry(
                agent_a=agent_name,
                agent_b=agent_name,
                rules_a=make_rules(base_rules=base_rules, simulation_budget=budget_a),
                rules_b=make_rules(base_rules=base_rules, simulation_budget=budget_b),
                total_games=total_games,
            )
            schedule.append(entry)

    return schedule


In [8]:
# Cell 4
def build_temp_schedule(
    agent_name,
    temps,
    history=10,
    total_games=100,
    base_rules=None,
):
    schedule = []

    for i, temp_b in enumerate(temps):
        predecessors = get_predecessors(temps, i, history=history)

        for temp_a in predecessors:
            entry = make_entry(
                agent_a=agent_name,
                agent_b=agent_name,
                rules_a=make_rules(base_rules=base_rules, initial_temp=temp_a, temp_decay_turn = 10000),
                rules_b=make_rules(base_rules=base_rules, initial_temp=temp_b, temp_decay_turn = 10000),
                total_games=total_games,
            )
            schedule.append(entry)

    return schedule


In [35]:
# Cell 5 — method 1: разные агенты, один бюджет
versions = list(range(1, 322, 10))
agents = [f"agent_v{v}.pth" for v in versions]

schedule_agents = build_agent_schedule(
    agents=agents,
    simulation_budget=200,
    history=10,
    total_games=800,
    base_rules=DEFAULT_RULES,
)

preview_schedule(schedule_agents, limit=10)
summarize_schedule(schedule_agents)
save_schedule_jsonl(schedule_agents, "./shared_data/tournament_schedule_agents.jsonl")


agent_v1.pth         vs agent_v11.pth        | budget_a=200 | budget_b=200 | games=800
agent_v1.pth         vs agent_v21.pth        | budget_a=200 | budget_b=200 | games=800
agent_v11.pth        vs agent_v21.pth        | budget_a=200 | budget_b=200 | games=800
agent_v1.pth         vs agent_v31.pth        | budget_a=200 | budget_b=200 | games=800
agent_v11.pth        vs agent_v31.pth        | budget_a=200 | budget_b=200 | games=800
agent_v21.pth        vs agent_v31.pth        | budget_a=200 | budget_b=200 | games=800
agent_v1.pth         vs agent_v41.pth        | budget_a=200 | budget_b=200 | games=800
agent_v11.pth        vs agent_v41.pth        | budget_a=200 | budget_b=200 | games=800
agent_v21.pth        vs agent_v41.pth        | budget_a=200 | budget_b=200 | games=800
agent_v31.pth        vs agent_v41.pth        | budget_a=200 | budget_b=200 | games=800
Итого матчей: 275
Итого игр: 220000
Сохранено: shared_data\tournament_schedule_agents.jsonl


In [36]:
# Cell 6 — method 2: один агент, разные бюджеты
budgets = [800, 700, 600, 500, 450, 400, 350, 300, 250, 200, 180, 160, 140, 120, 100, 80, 60, 40, 20, 10]

schedule_budgets = build_budget_schedule(
    agent_name="agent_v321.pth",   # сюда подставляешь нужный .pth
    budgets=budgets,
    history=10,
    total_games=800,
    base_rules=DEFAULT_RULES,
)

preview_schedule(schedule_budgets, limit=20)
summarize_schedule(schedule_budgets)
save_schedule_jsonl(schedule_budgets, "./shared_data/tournament_schedule_budgets.jsonl")


agent_v321.pth       vs agent_v321.pth       | budget_a=800 | budget_b=700 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=800 | budget_b=600 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=700 | budget_b=600 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=800 | budget_b=500 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=700 | budget_b=500 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=600 | budget_b=500 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=800 | budget_b=450 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=700 | budget_b=450 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=600 | budget_b=450 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=500 | budget_b=450 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=800 | budget_b=400 | games=800
agent_v321.pth       vs agent_v321.pth     

In [15]:
temps = [500, 250, 100, 50, 25, 10, 8, 6, 4, 3, 2, 1, 0.5, 0.3, 0.15, 0.1, 0.0]

schedule_temps = build_temp_schedule(
    agent_name="agent_v321.pth",   # сюда подставляешь нужный .pth
    temps=temps,
    history=8,
    total_games=800,
    base_rules=DEFAULT_RULES,
)

preview_schedule(schedule_temps, limit=20)
summarize_schedule(schedule_temps)
save_schedule_jsonl(schedule_temps, "./shared_data/tournament_schedule_temps.jsonl")

agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth       | budget_a=200 | budget_b=200 | games=800
agent_v321.pth       vs agent_v321.pth     